In [1]:
import os
import sys
from lca_algebraic import initProject
import pandas as pd
import numpy as np
import xarray as xr
import lca_algebraic as agb
from lca_algebraic import *  # version 1.0.5 
from lca_algebraic.stats import * 
import bw2data, bw2io
import bw2calc
from premise import *
import premise as prm
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import colormaps
matplotlib.style.use('seaborn-v0_8-whitegrid') #seaborn-v0_8-whitegrid  tableau-colorblind10

In [2]:
sys.path.append('/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/') 
from utils import *
from utils.setup import *
from utils.getAct import *
from utils.utils import *
#from utils.switchElec import *

modify the setup.py file for reading Parameter excel sheet. here we use 'data/p_file/p_tech_perSSP_Y.xlsx' as the technology parameter file.  
Define your chosen methods in the setup.py file. Current methods are:  [('IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT'), ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
Mapping premise_remind_DB to SSPx:  {'SSP1-PkBudg500': 'ssp119', 'SSP2-Base': 'ssp245', 'SSP5-Base': 'ssp585'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
pGWP100 are imported as:  [('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP585_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP119_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021

In [3]:
from utils_pGWP.pGWP_F2_V1 import *

In [4]:
bw2data.projects.set_current('iveo_v1')
mybio = bw2data.Database("biosphere3")
initProject("iveo_v1")

Biosphere database already present!!! No setup is needed


### init 

In [5]:
root_path = '/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/'
p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/p_tech_perSSP_Y.xlsx'))

In [6]:
pdb = bw.Database("iveo_Parameterized_v1")
print(len(pdb))
agb.loadParams()

384


[ParamRegistry] Param NMC622_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param NCA_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LFP_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LTO_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_NG_MJperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_elec_kWhperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param g_truck_vehicle_prod_lightweighting was already defined in '<project>' : overriding.
[ParamRegistry] Param battery_total_capacity_for_gtruck was already defined in '<project>' : overriding.
[ParamRegistry] Param biodiesel_UCO_perc was already defined in '<project>' : overriding.
[ParamRegistry] Pa

### get already created UPR

In [7]:
icevgtruck_finalallsspyear = [] 
for prm_name in ssp_scn:
    for year_t in ssp_yr: 
        bg_db_ssp_yr = bg_ei_ + prm_name + "_" + str(year_t)            # backgroun database (BD) to use for SSP[x], Year[t]
        copied_prm_UPR_suffix = "_from_" + prm_name + "_" + str(year_t) # copy UPR from BD to PDB for further use/modification if needed
        newUPR_suffix =  "_wh_" + prm_name + "_" + str(year_t)          # naming on any new UPRs created in PDB
        # final one unit of g_truck with lightweighting as Parameter: 
        gtruck_final  = agb.findActivity(
                db_name = PDB_NAME,
                name="V1A_g-truck ICEV-d Vehicle Production_" + newUPR_suffix)
        icevgtruck_finalallsspyear.append(gtruck_final)

len(icevgtruck_finalallsspyear)

9

### for ICEV, no TRL table needed

## calculate with the bat_tech_switch P and yearlyP, 
-   IPCC_GWP100 | premise_GWP100 | pGWP100

In [9]:
GWP0_list, GWP1_list, GWP2_list, SSP_list, Year_list, tech_list = [], [],[],[],[], []

for gtruck_final in icevgtruck_finalallsspyear:
    # need a matched YearlyParams for each SSPx:
    str1 = gtruck_final.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    
    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = p_file, s_name = ['V1A_g_truck'], years = [year_to_get_inP], SSP = ssp_to_get_inP) 
    #print(f"for SSP {ssp_to_get_inP}, year {year_to_get_inP}, P:  {yearlyParams}" ) 
        
    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = p_file, s_name = ['V1A_g_truck', 'V1B_bat_LSB_perkWh', 'V1B_bat_prmLIBs_perkWh'], years = [year_to_get_inP], SSP = ssp_to_get_inP) 
    #print(f"for SSP {ssp_to_get_inP}, year {year_to_get_inP}, P:  {yearlyParams}" )

    """ prepare methods: GWP1_list using static GWP,  GWP2_list using the pGWP100 for the matched v_year """ 
    gwp_spx_yt = get_my_pGWP100(ssp = ssp_to_get_inP, my = year_to_get_inP, metric = 'pGWP') 
    methods_touse = [chosen_methods[0], chosen_methods[1], gwp_spx_yt] # IPCCGWP100, premiseGWP, pGWP100_per_premiseGWP

    ### add a dummy_tech for ICEV 
    for tech in ['dummy_tech'] : 
        df= agb.multiLCAAlgebric(models=[
                (gtruck_final, 1)],  #act_test
                methods = methods_touse,   
                **yearlyParams
            )
        print(f"end of running LCA calc for SSP {ssp_to_get_inP} and year {year_to_get_inP} for {gtruck_final} ")
        display(df)
            
        # prepare for final DF: 
        GWP0 = df.iloc[:,0].values[0]
        GWP1 = df.iloc[:,1].values[0]
        GWP2 = df.iloc[:,2].values[0]
        GWP0_list.append(GWP0)
        GWP1_list.append(GWP1)
        GWP2_list.append(GWP2) 
        SSP_list.append(ssp_to_get_inP) 
        Year_list.append(year_to_get_inP)
        tech_list.append(tech)
         

end of running LCA calc for SSP ssp119 and year 2030 for 'V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP1-PkBudg500_2030' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP1-PkBudg500_2030,98438.795525,101192.073576,99125.053226


end of running LCA calc for SSP ssp119 and year 2040 for 'V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP1-PkBudg500_2040' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP1-PkBudg500_2040,87588.25938,88860.703452,87556.976391


end of running LCA calc for SSP ssp119 and year 2050 for 'V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP1-PkBudg500_2050' (one_truck, GLO, None) 


...

,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP5-Base_2030,119818.908461,122824.628623,119240.283299


end of running LCA calc for SSP ssp585 and year 2040 for 'V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP5-Base_2040' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP5-Base_2040,116338.825106,119487.61151,116288.67107


end of running LCA calc for SSP ssp585 and year 2050 for 'V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP5-Base_2050' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_g-truck ICEV-d Vehicle Production__wh__remind_SSP5-Base_2050,95006.381889,98170.079895,96562.311548


### prepare DF and DS 

In [14]:
# keep the v_module name same as BEV for "V1AB_init", for final benchmark if needed 
d1 = get_plcia_wh_V1A_ICEV(SSP_list, Year_list, tech_list, GWP0_list, GWP1_list, GWP2_list) 
d1

IPCC2021_default    premise_GWP  \
SSP    v_module  v_year tech_list                                     
ssp119 V1AB_init 2030   dummy_tech      98438.795525  101192.073576   
                 2040   dummy_tech      87588.259380   88860.703452   
                 2050   dummy_tech      84479.855746   83778.524593   
ssp245 V1AB_init 2030   dummy_tech     114930.409045  118034.668417   
                 2040   dummy_tech     104599.685668  107803.595647   
                 2050   dummy_tech      97562.766626  100837.469129   
ssp585 V1AB_init 2030   dummy_tech     119818.908461  122824.628623   
                 2040   dummy_tech     116338.825106  119487.611510   
                 2050   dummy_tech      95006.381889   98170.079895   

                                          pGWP100  
SSP    v_module  v_year tech_list                  
ssp119 V1AB_init 2030   dummy_tech   99125.053226  
                 2040   dummy_tech   87556.976391  
                 2050   dummy_tech   82663.226564  
ssp245 V1AB_init 2030   dummy_tech  115226.970692  
                 2040   dummy_tech  106102.075150  
                 2050   dummy_tech   99868.612059  
ssp585 V1AB_init 2030   dummy_tech  119240.283299  
                 2040   dummy_tech  116288.671070  
                 2050   dummy_tech   96562.311548

In [15]:
icev_init_ds = get_plcia_ds(d1)
icev_init_ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_module: 1, v_year: 3, tech_list: 1,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) object 'ssp119' 'ssp245' 'ssp585'
  * v_module         (v_module) object 'V1AB_init'
  * v_year           (v_year) int64 2030 2040 2050
  * tech_list        (tech_list) object 'dummy_tech'
  * Impact_category  (Impact_category) object 'IPCC2021_default' ... 'premise...
Data variables:
    value            (SSP, v_module, v_year, tech_list, Impact_category) float64 ...

In [17]:
d1.to_excel('pGWP_module_output/excel_output/V1AB_init_ICEVGT_pGWP100.xlsx' )

In [18]:
icev_init_ds.to_netcdf('pGWP_module_output/V1AB_init_ICEVGT_pGWP100.nc')